In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 02 — Transformação Silver
# MAGIC
# MAGIC **Objetivo:** limpar, tipar e padronizar os dados vindos da Bronze. Cada transformação
# MAGIC abaixo está documentada: o que foi feito, por que e qual o impacto.
# MAGIC
# MAGIC Também selecionamos, de `application_train_raw` (122 colunas), apenas os campos relevantes
# MAGIC para as perguntas de negócio do MVP — colunas de metadados de imóvel (`*_AVG`, `*_MODE`,
# MAGIC `*_MEDI`) foram descartadas por não se relacionarem ao objetivo e por terem altíssima taxa
# MAGIC de nulos (>50% em várias delas).

# COMMAND ----------

CATALOG = "mvp_creditomvp_credito.silver.historico_credito"

# COMMAND ----------

# MAGIC %md
# MAGIC ## `silver.clientes` (a partir de `bronze.application_train_raw`)
# MAGIC
# MAGIC **Transformações aplicadas:**
# MAGIC 1. Remoção de duplicatas por `SK_ID_CURR` (chave de cliente).
# MAGIC 2. `DAYS_BIRTH` (negativo, em dias) convertido para `idade` (anos, positivo).
# MAGIC 3. `DAYS_EMPLOYED` contém uma anomalia conhecida deste dataset: o valor `365243`
# MAGIC    (equivalente a ~1000 anos) é usado como *placeholder* para clientes sem vínculo
# MAGIC    empregatício formal (aposentados, desempregados). Convertemos esse valor para nulo e
# MAGIC    criamos uma flag `sem_vinculo_empregaticio` para não perder essa informação.
# MAGIC 4. Colunas selecionadas e renomeadas para português, focando no que é relevante às
# MAGIC    perguntas de negócio (perfil demográfico, financeiro e scores externos de crédito).
# MAGIC 5. Filtragem de registros sem `AMT_INCOME_TOTAL` (renda é atributo central da análise).

# COMMAND ----------

from pyspark.sql import functions as F

df_bronze_app = spark.table(f"{CATALOG}.bronze.application_train_raw")

df_silver_clientes = (
    df_bronze_app
    .dropDuplicates(["SK_ID_CURR"])
    .withColumn("idade", (F.abs(F.col("DAYS_BIRTH")) / 365.25).cast("int"))
    .withColumn(
        "sem_vinculo_empregaticio",
        F.when(F.col("DAYS_EMPLOYED") == 365243, True).otherwise(False)
    )
    .withColumn(
        "anos_empregado",
        F.when(F.col("DAYS_EMPLOYED") == 365243, None)
         .otherwise((F.abs(F.col("DAYS_EMPLOYED")) / 365.25).cast("int"))
    )
    .select(
        F.col("SK_ID_CURR").alias("sk_id_curr"),
        F.col("TARGET").alias("target"),
        F.col("NAME_CONTRACT_TYPE").alias("tipo_contrato"),
        F.col("CODE_GENDER").alias("genero"),
        F.col("FLAG_OWN_CAR").alias("possui_carro"),
        F.col("FLAG_OWN_REALTY").alias("possui_imovel"),
        F.col("CNT_CHILDREN").alias("qtd_filhos"),
        F.col("AMT_INCOME_TOTAL").alias("renda_total"),
        F.col("AMT_CREDIT").alias("valor_credito_solicitado"),
        F.col("AMT_ANNUITY").alias("valor_parcela"),
        F.col("NAME_INCOME_TYPE").alias("tipo_renda"),
        F.col("NAME_EDUCATION_TYPE").alias("escolaridade"),
        F.col("NAME_FAMILY_STATUS").alias("estado_civil"),
        F.col("NAME_HOUSING_TYPE").alias("tipo_moradia"),
        F.col("OCCUPATION_TYPE").alias("ocupacao"),
        F.col("CNT_FAM_MEMBERS").alias("qtd_membros_familia"),
        F.col("REGION_RATING_CLIENT").alias("rating_regiao"),
        F.col("EXT_SOURCE_1").alias("score_externo_1"),
        F.col("EXT_SOURCE_2").alias("score_externo_2"),
        F.col("EXT_SOURCE_3").alias("score_externo_3"),
        "idade",
        "anos_empregado",
        "sem_vinculo_empregaticio",
    )
    .filter(F.col("renda_total").isNotNull())
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## `silver.historico_credito` (a partir de `bronze.bureau_raw`)
# MAGIC
# MAGIC **Transformações aplicadas:**
# MAGIC 1. Remoção de duplicatas exatas.
# MAGIC 2. Filtragem de registros sem `SK_ID_CURR` válido (chave de junção com clientes).
# MAGIC 3. Tratamento de nulos em `AMT_CREDIT_SUM_DEBT` (dívida atual) e `AMT_CREDIT_SUM_OVERDUE`
# MAGIC    (valor em atraso): ausência de valor é tratada como 0, já que a ausência aqui
# MAGIC    significa que não há dívida/atraso reportado, não que o dado esteja faltando por erro.
# MAGIC 4. `DAYS_CREDIT` convertido para referência de tempo mais legível (mantido em dias
# MAGIC    relativo à data de referência, conforme padrão do dataset).
# MAGIC 5. Colunas renomeadas e selecionadas.

# COMMAND ----------

df_bronze_bureau = spark.table(f"{CATALOG}.bronze.bureau_raw")

df_silver_bureau = (
    df_bronze_bureau
    .dropDuplicates()
    .filter(F.col("SK_ID_CURR").isNotNull())
    .withColumn("AMT_CREDIT_SUM_DEBT", F.coalesce(F.col("AMT_CREDIT_SUM_DEBT"), F.lit(0.0)))
    .withColumn("AMT_CREDIT_SUM_OVERDUE", F.coalesce(F.col("AMT_CREDIT_SUM_OVERDUE"), F.lit(0.0)))
    .select(
        F.col("SK_ID_CURR").alias("sk_id_curr"),
        F.col("SK_ID_BUREAU").alias("sk_id_bureau"),
        F.col("CREDIT_ACTIVE").alias("status_credito"),
        F.col("CREDIT_TYPE").alias("tipo_credito"),
        F.col("DAYS_CREDIT").alias("dias_desde_abertura"),
        F.col("DAYS_CREDIT_ENDDATE").alias("dias_ate_vencimento"),
        F.col("AMT_CREDIT_SUM").alias("valor_credito"),
        F.col("AMT_CREDIT_SUM_DEBT").alias("valor_divida_atual"),
        F.col("AMT_CREDIT_SUM_OVERDUE").alias("valor_em_atraso"),
        F.col("CREDIT_DAY_OVERDUE").alias("dias_atraso"),
    )
)

# COMMAND ----------

# MAGIC %md
# MAGIC ### Persistência das tabelas Silver + comentários de catálogo

# COMMAND ----------

df_silver_clientes.write.mode("overwrite").saveAsTable(f"{CATALOG}.silver.clientes")
df_silver_bureau.write.mode("overwrite").saveAsTable(f"{CATALOG}.silver.historico_credito")

spark.sql(f"""
    COMMENT ON TABLE {CATALOG}.silver.clientes IS
    '1 linha por cliente, com atributos demográficos, financeiros e scores de crédito externos, já limpos e tipados. Origem: bronze.application_train_raw.'
""")
spark.sql(f"""
    COMMENT ON TABLE {CATALOG}.silver.historico_credito IS
    'Histórico de créditos do cliente em outras instituições (bureau), limpo e deduplicado. Múltiplas linhas por cliente. Origem: bronze.bureau_raw.'
""")

comentarios_clientes = {
    "sk_id_curr": "Identificador único do cliente. Domínio: inteiro > 0, chave primária.",
    "target": "Indicador de inadimplência. Domínio: 0 = adimplente, 1 = inadimplente (teve dificuldade de pagamento).",
    "tipo_contrato": 'Tipo de contrato de crédito solicitado. Domínio: "Cash loans", "Revolving loans".',
    "genero": 'Gênero do cliente. Domínio: "M", "F", "XNA" (não informado).',
    "possui_carro": 'Se o cliente possui carro. Domínio: "Y", "N".',
    "possui_imovel": 'Se o cliente possui imóvel. Domínio: "Y", "N".',
    "qtd_filhos": "Número de filhos do cliente. Domínio: inteiro >= 0.",
    "renda_total": "Renda total anual declarada pelo cliente, em valor monetário. Domínio: > 0 (possui outliers extremos, ver etapa de qualidade).",
    "valor_credito_solicitado": "Valor do crédito solicitado nesta aplicação. Domínio: > 0.",
    "valor_parcela": "Valor da parcela (anuidade) do crédito solicitado. Domínio: > 0, pode ser nulo.",
    "tipo_renda": 'Categoria da fonte de renda. Domínio: ex. "Working", "Pensioner", "Commercial associate".',
    "escolaridade": 'Nível de escolaridade do cliente. Domínio: ex. "Higher education", "Secondary / secondary special".',
    "estado_civil": 'Estado civil do cliente. Domínio: ex. "Married", "Single / not married".',
    "tipo_moradia": 'Tipo de moradia do cliente. Domínio: ex. "House / apartment", "With parents".',
    "ocupacao": "Ocupação profissional do cliente. Domínio: categórico, pode ser nulo (não informado).",
    "qtd_membros_familia": "Número de membros da família do cliente. Domínio: inteiro >= 1.",
    "rating_regiao": "Rating da região de moradia do cliente, atribuído pela instituição. Domínio: inteiro de 1 (melhor) a 3 (pior).",
    "score_externo_1": "Score de crédito normalizado vindo de fonte externa 1. Domínio: decimal entre 0 e 1, pode ser nulo.",
    "score_externo_2": "Score de crédito normalizado vindo de fonte externa 2. Domínio: decimal entre 0 e 1, pode ser nulo.",
    "score_externo_3": "Score de crédito normalizado vindo de fonte externa 3. Domínio: decimal entre 0 e 1, pode ser nulo.",
    "idade": "Idade do cliente em anos, derivada de DAYS_BIRTH. Domínio: inteiro, tipicamente entre 18 e 70.",
    "anos_empregado": "Anos de vínculo empregatício, derivado de DAYS_EMPLOYED. Nulo quando sem_vinculo_empregaticio = true.",
    "sem_vinculo_empregaticio": "Flag que identifica clientes sem vínculo empregatício formal (aposentados/desempregados), derivada do tratamento da anomalia DAYS_EMPLOYED = 365243.",
}
# escapa aspas simples (') dobrando-as, conforme sintaxe SQL, antes de montar o COMMENT
for coluna, comentario in comentarios_clientes.items():
    comentario_sql = comentario.replace("'", "''")
    spark.sql(f"ALTER TABLE {CATALOG}.silver.clientes ALTER COLUMN {coluna} COMMENT '{comentario_sql}'")

comentarios_bureau = {
    "sk_id_curr": "Identificador do cliente. Chave estrangeira para silver.clientes.",
    "sk_id_bureau": "Identificador único do crédito reportado ao bureau. Domínio: inteiro > 0, chave primária desta tabela.",
    "status_credito": 'Situação do crédito. Domínio: "Active", "Closed", "Sold", "Bad debt".',
    "tipo_credito": 'Tipo de crédito reportado. Domínio: ex. "Consumer credit", "Credit card", "Mortgage".',
    "dias_desde_abertura": "Dias entre a abertura do crédito e a data de referência da aplicação atual. Domínio: negativo (passado).",
    "dias_ate_vencimento": "Dias entre a data de referência e o vencimento previsto do crédito. Pode ser nulo.",
    "valor_credito": "Valor total do crédito concedido. Domínio: >= 0.",
    "valor_divida_atual": "Saldo devedor atual do crédito. Domínio: >= 0 (0 quando não informado/quitado).",
    "valor_em_atraso": "Valor em atraso no momento do reporte. Domínio: >= 0 (0 quando sem atraso).",
    "dias_atraso": "Dias de atraso no momento do reporte. Domínio: inteiro >= 0.",
}
for coluna, comentario in comentarios_bureau.items():
    comentario_sql = comentario.replace("'", "''")
    spark.sql(f"ALTER TABLE {CATALOG}.silver.historico_credito ALTER COLUMN {coluna} COMMENT '{comentario_sql}'")

# COMMAND ----------

print("silver.clientes:", spark.table(f"{CATALOG}.silver.clientes").count(), "linhas")
print("silver.historico_credito:", spark.table(f"{CATALOG}.silver.historico_credito").count(), "linhas")
display(spark.table(f"{CATALOG}.silver.clientes").limit(5))

silver.clientes: 307511 linhas
silver.historico_credito: 1716428 linhas


sk_id_curr,target,tipo_contrato,genero,possui_carro,possui_imovel,qtd_filhos,renda_total,valor_credito_solicitado,valor_parcela,tipo_renda,escolaridade,estado_civil,tipo_moradia,ocupacao,qtd_membros_familia,rating_regiao,score_externo_1,score_externo_2,score_externo_3,idade,anos_empregado,sem_vinculo_empregaticio
100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,State servant,Higher education,Married,House / apartment,Core staff,2.0,1,0.3112673113812225,0.6222457752555098,null,45,3,false
100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,Working,Secondary / secondary special,Single / not married,House / apartment,Laborers,1.0,2,null,0.5559120833904428,0.7295666907060153,52,0,false
100008,0,Cash loans,M,N,Y,0,99000.0,490495.5,27517.5,State servant,Secondary / secondary special,Married,House / apartment,Laborers,2.0,2,null,0.3542247319929012,0.6212263380626669,46,4,false
100070,0,Cash loans,M,Y,Y,0,540000.0,1227901.5,46899.0,Working,Higher education,Widow,House / apartment,Managers,1.0,1,null,0.6535972121081295,0.33928769990891394,56,5,false
100102,0,Cash loans,F,N,N,1,126000.0,327024.0,10264.5,Working,Secondary / secondary special,Single / not married,House / apartment,Laborers,2.0,2,0.4147940693543055,0.6611083077510267,0.4776491548517548,39,0,false
